# UniRef Table Validation
Checks row counts, schema, sample data, and internal consistency for all registered UniRef namespaces.

In [ ]:
# get_spark_session and create_namespace_if_not_exists are pre-loaded in the hub
spark = get_spark_session("UniRefValidation", tenant_name="refdata")

In [ ]:
# All namespaces to validate — comment out uniref100_2026_01 until it finishes loading
NAMESPACES = [
    "refdata_uniref50_2025_03",
    "refdata_uniref90_2025_03",
    "refdata_uniref100_2025_03",
    "refdata_uniref50_2026_01",
    "refdata_uniref90_2026_01",
    # "refdata_uniref100_2026_01",  # uncomment when loading completes
]
TABLES = ["cluster", "entity", "entity_x_source_file", "clustermember"]

## 1. Row Counts

In [ ]:
import pandas as pd

counts = []
for ns in NAMESPACES:
    for table in TABLES:
        try:
            n = spark.sql(f"SELECT COUNT(*) FROM {ns}.{table}").collect()[0][0]
            counts.append({"namespace": ns, "table": table, "rows": n})
        except Exception as e:
            counts.append({"namespace": ns, "table": table, "rows": f"ERROR: {e}"})

df_counts = pd.DataFrame(counts)
print(df_counts.to_string(index=False))

## 2. Internal Consistency
- `cluster` and `entity` should have the same row count (1:1 mapping)
- Every `cluster` should have exactly 1 representative member in `clustermember`
- `entity_x_source_file` should have the same count as `cluster`

In [ ]:
# Reuse counts already computed above; only run 2 queries per namespace instead of 6
def get_count(ns, table):
    row = df_counts[(df_counts.namespace == ns) & (df_counts.table == table)]["rows"]
    return row.values[0] if len(row) else "missing"

for ns in NAMESPACES:
    print(f"\n=== {ns} ===")
    try:
        n_cluster = get_count(ns, "cluster")
        n_entity  = get_count(ns, "entity")
        n_exsf    = get_count(ns, "entity_x_source_file")

        # Combine all clustermember checks into a single scan
        cm = spark.sql(f"""
            SELECT
                SUM(CASE WHEN is_representative THEN 1 ELSE 0 END) AS n_repr,
                SUM(CASE WHEN entity_id IS NULL  THEN 1 ELSE 0 END) AS n_null_eid
            FROM {ns}.clustermember
        """).collect()[0]
        n_repr, n_null_eid = cm["n_repr"], cm["n_null_eid"]

        n_null_cid = spark.sql(
            f"SELECT COUNT(*) FROM {ns}.cluster WHERE cluster_id IS NULL"
        ).collect()[0][0]

        print(f"  cluster == entity:               {n_cluster == n_entity} ({n_cluster:,} vs {n_entity:,})")
        print(f"  cluster == entity_x_source_file: {n_cluster == n_exsf} ({n_cluster:,} vs {n_exsf:,})")
        print(f"  cluster == repr members:         {n_cluster == n_repr} ({n_cluster:,} vs {n_repr:,})")
        print(f"  null cluster_ids:                {n_null_cid}")
        print(f"  null entity_ids in clustermember:{n_null_eid}")
    except Exception as e:
        print(f"  ERROR: {e}")


## 3. entity_id Prefix Distribution
Should see `uniprot:`, `uniref:`, `uniparc:` prefixes — no bare IDs.

In [ ]:
# Cheapest possible prefix check: fetch 5 entity_id values per namespace.
# No aggregation — just eyeball whether the prefix pattern looks right.
for ns in NAMESPACES:
    print(f"\n=== {ns}.clustermember entity_id sample ===")
    try:
        spark.sql(f"SELECT entity_id FROM {ns}.clustermember LIMIT 5").show(truncate=False)
    except Exception as e:
        print(f"  ERROR: {e}")


## 4. Sample Rows

In [ ]:
# Spot-check one namespace in detail
SAMPLE_NS = "refdata_uniref50_2026_01"

print(f"--- {SAMPLE_NS}.cluster ---")
spark.sql(f"SELECT * FROM {SAMPLE_NS}.cluster LIMIT 5").show(truncate=False)

print(f"--- {SAMPLE_NS}.entity ---")
spark.sql(f"SELECT * FROM {SAMPLE_NS}.entity LIMIT 5").show(truncate=False)

print(f"--- {SAMPLE_NS}.clustermember (representatives) ---")
spark.sql(f"SELECT * FROM {SAMPLE_NS}.clustermember WHERE is_representative = true LIMIT 5").show(truncate=False)

print(f"--- {SAMPLE_NS}.entity_x_source_file ---")
spark.sql(f"SELECT * FROM {SAMPLE_NS}.entity_x_source_file LIMIT 5").show(truncate=False)

## 5. Version Comparison (2025_03 vs 2026_01)
Cluster counts should be similar but slightly higher in 2026_01.

In [ ]:
# Derived from df_counts already computed in cell 1 — no extra queries
for variant in ["uniref50", "uniref90", "uniref100"]:
    print(f"\n{variant}:")
    for version in ["2025_03", "2026_01"]:
        ns = f"refdata_{variant}_{version}"
        row = df_counts[(df_counts.namespace == ns) & (df_counts.table == "cluster")]["rows"]
        if len(row) and not str(row.values[0]).startswith("ERROR"):
            print(f"  {version}: {int(row.values[0]):,} clusters")
        else:
            print(f"  {version}: not yet available")

## 6. Register uniref100 2026_01 (run after loading completes)

In [ ]:
# Run this cell only after uniref100 2026_01 finishes writing
# namespace = create_namespace_if_not_exists(spark, namespace="uniref100_2026_01", tenant_name="refdata")
# base = "s3a://cdm-lake/tenant-sql-warehouse/refdata/refdata_uniref100_2026_01.db"
# for table in ["cluster", "entity", "entity_x_source_file", "clustermember"]:
#     spark.sql(f"CREATE TABLE IF NOT EXISTS {namespace}.{table} USING DELTA LOCATION '{base}/{table}'")
#     count = spark.sql(f"SELECT COUNT(*) FROM {namespace}.{table}").collect()[0][0]
#     print(f"  {table}: {count:,} rows")
print("Uncomment and run once uniref100 2026_01 loading is complete.")